# Synthea Tutorial: Synthetic Patient Data Generation

This notebook provides a comprehensive tutorial on using the Synthea wrapper module to generate synthetic healthcare patient data.

## What is Synthea?

Synthea™ is an open-source synthetic patient population simulator that generates realistic synthetic patient records. It creates entire lifetimes of patient data, including:

- Demographics
- Medical history
- Medications
- Lab results
- Procedures
- Encounters
- And more...

**Reference**: [Synthea GitHub](https://github.com/synthetichealth/synthea)

## Prerequisites

- Java 11 or newer installed
- Python environment with required packages
- Internet connection (for initial JAR download)


In [ ]:
%load_ext autoreload
%autoreload 2
%load_ext rpy2.ipython
# Import the Synthea wrapper
# If package is installed: pip install -e .
# Otherwise, add parent directory to path for development
import sys
import os

# Add parent directory to path if package not installed
if 'AoU' not in sys.modules:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from AoU.phenome.synthea import SyntheaRunner, SyntheaConfig, download_synthea_jar, convert_synthea_to_omop


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✓ Synthea module imported successfully


## 1. Initialization

The `SyntheaRunner` class handles downloading the Synthea JAR file (if needed) and provides methods to run simulations.

### Basic Initialization


In [3]:
# Initialize the Synthea runner
# This will automatically download the JAR file if it doesn't exist
runner = SyntheaRunner()

print(f"Synthea JAR location: {runner.jar_path}")
print(f"Java executable: {runner.java_executable}")


Using existing Synthea JAR: /home/schilder/.cache/synthea/synthea-with-dependencies.jar
Synthea JAR location: /home/schilder/.cache/synthea/synthea-with-dependencies.jar
Java executable: java


### Custom Initialization

You can specify custom paths and settings:


In [4]:
# Example: Custom cache directory
# runner = SyntheaRunner(
#     cache_dir="./synthea_cache",  # Where to store the JAR
#     java_executable="java"         # Path to Java (default: "java")
# )

# Example: Use existing JAR file
# runner = SyntheaRunner(
#     jar_path="/path/to/existing/synthea-with-dependencies.jar"
# )


## 2. Quick Start: Simple Simulation

The easiest way to generate synthetic patients is using the `run_quick()` method:


In [5]:
# Generate 10 patients in Massachusetts
result = runner.run_quick(
    population_size=10,
    state="Massachusetts",
    seed=42,  # For reproducibility
    output_dir="output/synthea_quick"
)

print(f"\nReturn code: {result['returncode']}")
print(f"Output directory: {result['output_dir']}")
print(f"\nCommand executed:")
print(result['command'])
# print(result["stdout"])



$ java -jar /home/schilder/.cache/synthea/synthea-with-dependencies.jar -s 42 -p 10 Massachusetts

Return code: 0
Output directory: /home/schilder/projects/AoU/notebooks/output/synthea_quick

Command executed:
java -jar /home/schilder/.cache/synthea/synthea-with-dependencies.jar -s 42 -p 10 Massachusetts


## 3. Full Configuration with SyntheaConfig

For more control, use the `SyntheaConfig` dataclass with the `run()` method:


In [11]:
# Create a detailed configuration
config = SyntheaConfig(
    # Population settings
    population_size=50,
    seed=12345,
    reference_date="20200101",  # Format: YYYYMMDD
    
    # Demographics
    gender="F",  # "M", "F", or None for both
    min_age=25,
    max_age=65,
    
    # Location
    state="California",
    city="San Francisco",
    
    # Output
    output_dir="output/synthea_california"
)

# Run the simulation
result = runner.run(config, verbose=True)

if result['returncode'] == 0:
    print("\n✓ Simulation completed successfully!")
else:
    print(f"\n✗ Simulation failed with return code {result['returncode']}")
    print(f"Error: {result['stderr']}")



$ java -jar /home/schilder/.cache/synthea/synthea-with-dependencies.jar -s 12345 -r 20200101 -p 50 -g F -a 25-65 California 'San Francisco'

✓ Simulation completed successfully!


## 4. Convenience Methods

The wrapper provides several convenience methods for common use cases:

### 4.1 Custom Location


In [12]:
# Generate patients in a specific city
result = runner.run_custom_location(
    state="Washington",
    city="Seattle",
    population_size=25,
    seed=999,
    output_dir="output/synthea_seattle"
)

print(f"Generated patients in Seattle, WA")
print(f"Output: {result['output_dir']}")



$ java -jar /home/schilder/.cache/synthea/synthea-with-dependencies.jar -s 999 -p 25 Washington Seattle
Generated patients in Seattle, WA
Output: /home/schilder/projects/AoU/notebooks/output/synthea_seattle


### 4.2 Age-Specific Population


In [13]:
# Generate only pediatric patients (ages 0-18)
result = runner.run_age_specific(
    min_age=0,
    max_age=18,
    population_size=30,
    state="Massachusetts",
    seed=456,
    output_dir="output/synthea_pediatric"
)

print(f"Generated pediatric population (ages 0-18)")



$ java -jar /home/schilder/.cache/synthea/synthea-with-dependencies.jar -s 456 -p 30 -a 0-18 Massachusetts
Generated pediatric population (ages 0-18)


## 5. Advanced Configuration

### 5.1 Using Custom Config Files

Synthea supports custom configuration files for fine-tuned control:


In [14]:
# Example: Using a custom config file
# config = SyntheaConfig(
#     population_size=100,
#     config_file="./custom_synthea.conf",
#     output_dir="output/synthea_custom"
# )
# result = runner.run(config)

# Note: You need to create the config file first
# See Synthea documentation for config file format


### 5.2 Custom Modules Directory

You can use custom Synthea modules:


In [15]:
# Example: Using custom modules
# config = SyntheaConfig(
#     population_size=50,
#     modules_dir="./custom_modules",
#     output_dir="output/synthea_custom_modules"
# )
# result = runner.run(config)


### 5.3 Exporter Flags

Synthea supports various exporters (FHIR, CSV, etc.). You can configure them using exporter flags:


In [16]:
# Example: Enable FHIR US Core IG exporter
config = SyntheaConfig(
    population_size=20,
    state="Massachusetts",
    seed=789,
    output_dir="output/synthea_fhir",
    exporter_flags={
        "exporter.fhir.use_us_core_ig": "true",
        "exporter.csv.export": "true"
    }
)

# result = runner.run(config)
print("Example configuration with FHIR US Core IG exporter")


Example configuration with FHIR US Core IG exporter


## 6. Working with Output Files

Synthea generates various output files depending on the exporters enabled. Let's explore what gets generated:


In [4]:
import os
from pathlib import Path

# Check output directory
output_dir = "output/synthea_quick"
if os.path.exists(output_dir):
    print(f"Files in {output_dir}:")
    for file in sorted(os.listdir(output_dir)):
        file_path = os.path.join(output_dir, file)
        size = os.path.getsize(file_path) / 1024  # Size in KB
        print(f"  - {file} ({size:.2f} KB)")
else:
    print(f"Output directory {output_dir} does not exist yet.")
    print("Run a simulation first to generate files.")


Files in output/synthea_quick:
  - output (0.04 KB)


### Common Output Formats

Synthea can generate data in multiple formats:

- **CSV**: Tabular data files
- **FHIR**: HL7 FHIR resources (JSON)
- **JSON**: Generic JSON format
- **CCDA**: Clinical Document Architecture

By default, Synthea generates CSV files. To enable other formats, use exporter flags.


In [5]:
# Example: Load and inspect CSV output (if available)
import pandas as pd

csv_files = []
if os.path.exists(output_dir):
    csv_files = [f for f in os.listdir(output_dir) if f.endswith('.csv')]

if csv_files:
    print(f"Found {len(csv_files)} CSV files:")
    for csv_file in csv_files[:5]:  # Show first 5
        print(f"  - {csv_file}")
        
    # Example: Load patients CSV
    patients_file = os.path.join(output_dir, "patients.csv")
    if os.path.exists(patients_file):
        df = pd.read_csv(patients_file, nrows=5)  # Load first 5 rows
        print(f"\nSample from patients.csv:")
        print(df.head())
else:
    print("No CSV files found. Run a simulation to generate data.")


No CSV files found. Run a simulation to generate data.


## 7. Getting Help

You can view Synthea's built-in help:


In [6]:
# Display Synthea help
help_text = runner.show_help()
print(help_text)


Usage: run_synthea [options] [state [city]]
Options: [-s seed] [-cs clinicianSeed] [-p populationSize]
         [-ps singlePersonSeed]
         [-r referenceDate as YYYYMMDD]
         [-e endDate as YYYYMMDD]
         [-g gender] [-a minAge-maxAge]
         [-o overflowPopulation]
         [-c localConfigFilePath]
         [-d localModulesDirPath]
         [-i initialPopulationSnapshotPath]
         [-u updatedPopulationSnapshotPath]
         [-t updateTimePeriodInDays]
         [-f fixedRecordPath]
         [-k keepMatchingPatientsPath]
         [--config*=value]
          * any setting from src/main/resources/synthea.properties
Examples:
run_synthea Massachusetts
run_synthea Alaska Juneau
run_synthea -s 12345
run_synthea -p 1000
run_synthea -s 987 Washington Seattle
run_synthea -s 21 -p 100 Utah "Salt Lake City"
run_synthea -g M -a 60-65
run_synthea -p 10 --exporter.fhir.export=true
run_synthea --exporter.baseDirectory="./output_tx/" Texas


## 8. Complete Example: Generating a Research Dataset

Here's a complete example for generating a dataset suitable for research:


In [7]:
# Generate a research-ready dataset
research_config = SyntheaConfig(
    # Generate 1000 patients for statistical power
    population_size=1000,
    
    # Use a fixed seed for reproducibility
    seed=20240101,
    
    # Reference date for the simulation
    reference_date="20200101",
    
    # Include all genders
    gender=None,
    
    # Adult population (18-80 years)
    min_age=18,
    max_age=80,
    
    # Geographic location
    state="Massachusetts",
    
    # Enable multiple exporters
    exporter_flags={
        "exporter.csv.export": "true",
        "exporter.fhir.export": "true",
        "exporter.fhir.use_us_core_ig": "true"
    },
    
    # Output directory
    output_dir="output/synthea_research_dataset"
)

print("Configuration created:")
print(f"  Population: {research_config.population_size}")
print(f"  Age range: {research_config.min_age}-{research_config.max_age}")
print(f"  Location: {research_config.state}")
print(f"  Output: {research_config.output_dir}")

# Uncomment to run:
# result = runner.run(research_config, verbose=True)
# if result['returncode'] == 0:
#     print("\n✓ Research dataset generated successfully!")
#     print(f"Files available in: {result['output_dir']}")


Configuration created:
  Population: 1000
  Age range: 18-80
  Location: Massachusetts
  Output: output/synthea_research_dataset


## 9. Tips and Best Practices

### Reproducibility
- **Always use seeds**: Set a `seed` parameter for reproducible results
- **Document your configuration**: Save your `SyntheaConfig` for future reference

### Performance
- **Start small**: Test with small populations (10-50) before running large simulations
- **Output directory**: Use separate directories for different experiments
- **Resource usage**: Large populations (1000+) can take significant time and disk space

### Data Quality
- **Validate output**: Check that output files are generated correctly
- **Review demographics**: Ensure the generated population matches your expectations
- **Multiple runs**: Consider running multiple simulations with different seeds for robustness

### Common Issues
1. **Java not found**: Ensure Java 11+ is installed and in PATH
2. **Out of memory**: Large populations may require increasing Java heap size
3. **Missing output**: Check that the output directory exists and is writable


In [9]:
# Example: Check system requirements
import subprocess

# Check Java version
try:
    result = subprocess.run(
        ["java", "-version"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    print("Java version check:")
    print(result.stdout)
except FileNotFoundError:
    print("⚠️ Java not found! Please install Java 11 or newer.")

# Check available disk space
import shutil
total, used, free = shutil.disk_usage(".")
print(f"\nDisk space:")
print(f"  Total: {total / (1024**3):.2f} GB")
print(f"  Free: {free / (1024**3):.2f} GB")
print(f"  Used: {used / (1024**3):.2f} GB")


Java version check:
openjdk version "17.0.14" 2025-01-21 LTS
OpenJDK Runtime Environment (Red_Hat-17.0.14.0.7-1) (build 17.0.14+7-LTS)
OpenJDK 64-Bit Server VM (Red_Hat-17.0.14.0.7-1) (build 17.0.14+7-LTS, mixed mode, sharing)


Disk space:
  Total: 7151.71 GB
  Free: 2026.75 GB
  Used: 5124.96 GB


[autoreload of src.phenome.synthea failed: Traceback (most recent call last):
  File "/home/schilder/.conda/envs/AoU/lib/python3.14/site-packages/IPython/extensions/autoreload.py", line 325, in check
    superreload(m, reload, self.old_objects)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/schilder/.conda/envs/AoU/lib/python3.14/site-packages/IPython/extensions/autoreload.py", line 580, in superreload
    module = reload(module)
  File "/home/schilder/.conda/envs/AoU/lib/python3.14/importlib/__init__.py", line 129, in reload
    _bootstrap._exec(spec, module)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 869, in _exec
  File "<frozen importlib._bootstrap_external>", line 755, in exec_module
  File "<frozen importlib._bootstrap_external>", line 893, in get_code
  File "<frozen importlib._bootstrap_external>", line 823, in source_to_code
  File "<frozen importlib._bootstrap>", line 491, in _call_with_frames_removed
  File "/home/schilder/pro

## 10. Integration with AoU Workflow

The Synthea-generated data can be integrated into your All of Us (AoU) research workflow:

1. **Generate synthetic data** using this wrapper
2. **Convert to OMOP format** if needed (Synthea has OMOP export capabilities)
3. **Load into your analysis pipeline** alongside real AoU data
4. **Use for testing** algorithms and pipelines before applying to real data

### Example: OMOP Export

Synthea can export data in OMOP CDM format, which is compatible with AoU:


In [6]:
# Example: Generate data with OMOP export
# Note: OMOP exporter may need to be enabled in Synthea config
omop_config = SyntheaConfig(
    population_size=100,
    state="Massachusetts",
    seed=42,
    output_dir="output/synthea_omop",
    exporter_flags={
        "exporter.omop.export": "true",
        "exporter.csv.export": "true"  # Also keep CSV for reference
    }
)

# result = runner.run(omop_config)
print("OMOP export configuration example")
print("Note: OMOP exporter availability depends on Synthea version")


OMOP export configuration example
Note: OMOP exporter availability depends on Synthea version


In [29]:
# Actually run the OMOP conversion example
# First, generate Synthea data with CSV export enabled
from pathlib import Path

omop_config = SyntheaConfig(
    population_size=50,  # Smaller for demo
    state="Massachusetts",
    seed=42,
    output_dir="output/synthea_omop",
    exporter_flags={
        "exporter.csv.export": "true"  # Ensure CSV export is enabled
    }
)

# Run the simulation
print("Generating Synthea data...")
result = runner.run(omop_config, force=0, verbose=True)

if result['returncode'] == 0:
    print("\n✓ Synthea data generated successfully!")
    
    # Check what CSV files were generated
    print(f"\nChecking for CSV files in: {result['output_dir']}")
    synthea_output = Path(result['output_dir'])
    
    # Synthea may create CSV files in a 'csv' subdirectory or directly in output
    csv_dirs_to_check = [
        synthea_output,
        synthea_output / "csv",
        synthea_output / "output" / "csv",
    ]
    
    csv_dir = None
    for check_dir in csv_dirs_to_check:
        if check_dir.exists():
            csv_files = list(check_dir.glob("*.csv"))
            if csv_files:
                csv_dir = check_dir
                print(f"  ✓ Found CSV files in: {csv_dir}")
                print(f"  Found {len(csv_files)} CSV files:")
                for csv_file in sorted(csv_files)[:10]:  # Show first 10
                    size = csv_file.stat().st_size / 1024  # KB
                    print(f"    - {csv_file.name} ({size:.2f} KB)")
                if len(csv_files) > 10:
                    print(f"    ... and {len(csv_files) - 10} more")
                break
    
    if csv_dir is None:
        print("  ⚠️  No CSV files found! Synthea may not have generated CSV output.")
        print("  Check that exporter.csv.export=true is set in Synthea configuration.")
    else:
        # Now convert to OMOP format
        print(f"\nConverting to OMOP CDM format...")
        
        omop_files = runner.convert_to_omop(
            synthea_output_dir=str(csv_dir),  # Use the directory where CSV files actually are
            omop_output_dir=os.path.join(result['output_dir'], "omop"),
            cdm_version="5.4",
            output_format="parquet",
            verbose=True
        )
        
        print(f"\n✓ Conversion complete!")
        print(f"\nGenerated {len(omop_files)} OMOP tables:")
        for table_name, file_path in sorted(omop_files.items()):
            file_size = Path(file_path).stat().st_size / 1024  # KB
            print(f"  - {table_name}: {file_path} ({file_size:.2f} KB)")
else:
    print(f"\n✗ Simulation failed with return code {result['returncode']}")
    print(f"Error: {result['stderr']}")


Generating Synthea data...

$ java -jar /home/schilder/.cache/synthea/synthea-with-dependencies.jar -s 42 -p 50 --exporter.csv.export=true Massachusetts

✓ Synthea data generated successfully!

Checking for CSV files in: /home/schilder/projects/AoU/notebooks/output/synthea_omop
  ✓ Found CSV files in: /home/schilder/projects/AoU/notebooks/output/synthea_omop/output/csv
  Found 18 CSV files:
    - allergies.csv (14.81 KB)
    - careplans.csv (50.17 KB)
    - claims.csv (4281.77 KB)
    - claims_transactions.csv (30508.74 KB)
    - conditions.csv (449.13 KB)
    - devices.csv (100.45 KB)
    - encounters.csv (1911.00 KB)
    - imaging_studies.csv (175.89 KB)
    - immunizations.csv (130.48 KB)
    - medications.csv (1439.13 KB)
    ... and 8 more

Converting to OMOP CDM format...
Converting Synthea CSV to OMOP CDM v5.4
Input directory: /home/schilder/projects/AoU/notebooks/output/synthea_omop/output/csv
Output directory: /home/schilder/projects/AoU/notebooks/output/synthea_omop/omop
  Co

## 12. Synthea to OMOP Conversion Methodology

The `convert_synthea_to_omop()` function converts Synthea CSV output to OMOP CDM format, inspired by the [OHDSI ETL-Synthea](https://github.com/OHDSI/ETL-Synthea) R package. Here's how it works:

### Conversion Process

The conversion follows these steps:

1. **Read Synthea CSV Files**: Loads Synthea-generated CSV files (patients, encounters, conditions, medications, procedures, observations)

2. **Map to OMOP CDM Tables**: Transforms Synthea data into OMOP CDM table structures:
   - **person**: Patient demographics (gender, birth date, etc.)
   - **visit_occurrence**: Healthcare encounters/visits
   - **condition_occurrence**: Diagnoses and conditions
   - **drug_exposure**: Medications and prescriptions
   - **procedure_occurrence**: Medical procedures
   - **measurement**: Numeric lab results and measurements
   - **observation**: Non-numeric observations
   - **death**: Death records
   - **observation_period**: Patient observation time periods
   - **cdm_source**: Metadata about the data source

3. **Field Mapping**: Maps Synthea fields to OMOP CDM fields:
   - Synthea `Id` → OMOP `person_id`
   - Synthea `GENDER` → OMOP `gender_concept_id` (8507=Male, 8532=Female)
   - Synthea `ENCOUNTERCLASS` → OMOP `visit_concept_id` (9201=Inpatient, 9202=Outpatient, 9203=Emergency)
   - Synthea `CODE` → OMOP `*_source_value` (preserves original SNOMED codes)
   - Date fields converted to OMOP date/datetime format

4. **Visit Linking**: Creates `visit_occurrence_id` mappings to link events (conditions, medications, etc.) to their associated visits

5. **Observation Splitting**: Separates observations into:
   - **measurement**: Numeric values (lab results, vitals)
   - **observation**: Non-numeric values (text observations)

6. **Output Generation**: Saves OMOP tables in Parquet or CSV format

### Key Features

- **Preserves Source Values**: Original Synthea codes (typically SNOMED) are preserved in `*_source_value` fields
- **Concept ID Placeholders**: `concept_id` fields are set to 0 and require vocabulary lookup for full OMOP compliance
- **Visit Relationships**: Automatically links events to their associated visits
- **Flexible Output**: Supports both Parquet (default) and CSV formats

### Important Notes

⚠️ **Concept Mapping**: The function sets `concept_id` fields to 0. For full OMOP compliance, you'll need to:
- Load OMOP vocabulary tables (concept, concept_relationship, etc.)
- Map Synthea source codes (in `*_source_value` fields) to OMOP concept_ids using vocabulary lookups
- Update the `concept_id` fields with the mapped values

⚠️ **Vocabulary Tables**: OMOP vocabulary tables (concept, vocabulary, concept_relationship, etc.) are not generated by this function and must be loaded separately from OMOP vocabulary releases.

### Usage

```python
# Method 1: Using the convenience method
omop_files = runner.convert_to_omop(
    synthea_output_dir="output/synthea_data",
    omop_output_dir="output/omop_data"
)

# Method 2: Using the standalone function
from AoU.phenome.synthea import convert_synthea_to_omop
omop_files = convert_synthea_to_omop(
    synthea_csv_dir="output/synthea_data",
    output_dir="output/omop_data",
    cdm_version="5.4",
    output_format="parquet"
)
```

### References

- [OHDSI ETL-Synthea R Package](https://github.com/OHDSI/ETL-Synthea)
- [OMOP CDM Documentation](https://ohdsi.github.io/CommonDataModel/)
- [Synthea GitHub](https://github.com/synthetichealth/synthea)


In [ ]:
# Generate 10 patients in Massachusetts
result = runner.run_quick(
    population_size=10,
    state="Massachusetts",
    seed=42,  # For reproducibility
    output_dir="output/synthea_quick"
)

print(f"\nReturn code: {result['returncode']}")
print(f"Output directory: {result['output_dir']}")
print(f"\nCommand executed:")
print(result['command'])
# print(result["stdout"])


## 11. Convert OMOP to Timeline


In [7]:
from AoU.phenome import timeline
from AoU.phenome.timeline import TimelineConfig, build_omop_timelines
import polars as pl

# Configure timeline generation for Synthea OMOP data
# The timeline module converts OMOP CDM tables into patient timelines
# organized by time bins (e.g., yearly, monthly, or by visit)

# Set up configuration
# Note: The timeline module expects OMOP data in: {omop_root}/{cohort}/OMOP/
# Our Synthea OMOP data is in: output/synthea_omop/omop/
# Configure timeline generation for Synthea OMOP data
# Note: Synthea only generates a subset of OMOP tables, so we exclude missing ones
timeline_cfg = TimelineConfig(
    cohort="synthea_omop",           # Cohort name (matches directory)
    omop_root="output",               # Root directory containing cohort folders
    bin_size="1y",                    # Time bin size: "1y", "1mo", "1w", etc.
    bin_mode="calendar",              # "calendar" or "visit"
    
    # Output paths (auto-generated if None)
    out_parquet="output/synthea_omop/timelines_calendar_1y.parquet",
    events_parquet="output/synthea_omop/events_unified.parquet",
    eras_parquet="output/synthea_omop/eras_periods.parquet",
    
    # Only include tables that Synthea actually generates
    # Available: condition_occurrence, drug_exposure, measurement, 
    #            observation, procedure_occurrence, death
    # Missing: device_exposure, specimen, survey_conduct, visit_detail
    main_event_tables=(
        "condition_occurrence",
        "drug_exposure",
        "measurement",
        "observation",
        "procedure_occurrence",
        "death",
    ),
    
    # Timeline formatting options
    markdown=True,                    # Generate markdown-formatted timelines
    visit_markdown_headings=True,     # Use visit-level headings in markdown
    include_interval_summary=True,     # Add YAML-style interval summaries
    include_src_name=True,            # Include source concept names
    
    # Force rebuild (0=use cache, 1=rebuild if config differs, 2=always rebuild)
    force=0
)

print("Timeline configuration:")
print(f"  OMOP directory: {timeline_cfg.omop_dir}")
print(f"  Output timeline: {timeline_cfg.out_parquet}")
print(f"  Bin size: {timeline_cfg.bin_size}")
print(f"  Bin mode: {timeline_cfg.bin_mode}")
print()

# Check if OMOP directory exists
import os
omop_dir = timeline_cfg.omop_dir
if os.path.exists(omop_dir):
    print(f"✓ Found OMOP directory: {omop_dir}")
    omop_files = [f for f in os.listdir(omop_dir) if f.endswith('.parquet')]
    print(f"  Found {len(omop_files)} OMOP tables")
    
    # Check for required concept table
    if 'concept.parquet' not in omop_files:
        print("\n⚠️  Note: 'concept.parquet' not found in OMOP directory.")
        print("   The timeline module requires a concept table for concept name lookups.")
        print("   For Synthea data, you may need to:")
        print("   1. Download OMOP vocabulary tables from OHDSI")
        print("   2. Or create a minimal concept table from source values")
        print("\n   For demonstration, we'll show the configuration setup.")
        print("   To actually run, you'll need to add a concept table.")
    else:
        print("✓ Concept table found - ready to build timelines")
else:
    print(f"✗ OMOP directory not found: {omop_dir}")
    print("  Please run the OMOP conversion example first (Section 10)")

print("\n" + "="*60)
print("Timeline Configuration Summary")
print("="*60)
print(f"Cohort: {timeline_cfg.cohort}")
print(f"OMOP Root: {timeline_cfg.omop_root}")
print(f"Time Binning: {timeline_cfg.bin_mode} mode, {timeline_cfg.bin_size} bins")
print(f"Output Format: {'Markdown' if timeline_cfg.markdown else 'Plain text'}")
print(f"Event Tables: {len(timeline_cfg.main_event_tables)} tables")
print(f"Era Tables: {len(timeline_cfg.era_tables)} tables")

Timeline configuration:
  OMOP directory: output/synthea_omop/OMOP
  Output timeline: output/synthea_omop/timelines_calendar_1y.parquet
  Bin size: 1y
  Bin mode: calendar

✓ Found OMOP directory: output/synthea_omop/OMOP
  Found 11 OMOP tables
✓ Concept table found - ready to build timelines

Timeline Configuration Summary
Cohort: synthea_omop
OMOP Root: output
Time Binning: calendar mode, 1y bins
Output Format: Markdown
Event Tables: 6 tables
Era Tables: 1 tables


In [8]:
# Example: Building timelines from OMOP data
# 
# To actually run the timeline conversion, you need a concept table.
# Here's how to create a minimal concept table from Synthea source values:

import polars as pl
import os

omop_dir = timeline_cfg.omop_dir
concept_path = os.path.join(omop_dir, "concept.parquet")

# Ensure the OMOP directory exists (TimelineConfig expects uppercase OMOP)
os.makedirs(omop_dir, exist_ok=True)

# If the expected OMOP directory doesn't have files, check for lowercase 'omop' directory
actual_omop_dir = os.path.join(timeline_cfg.omop_root, timeline_cfg.cohort, "omop")
if os.path.exists(actual_omop_dir):
    # Check if we need to copy files from lowercase to uppercase directory
    if not os.path.exists(omop_dir) or len([f for f in os.listdir(omop_dir) if f.endswith('.parquet')]) == 0:
        import shutil
        print(f"Copying OMOP files from {actual_omop_dir} to {omop_dir}...")
        for file in os.listdir(actual_omop_dir):
            if file.endswith('.parquet'):
                src = os.path.join(actual_omop_dir, file)
                dst = os.path.join(omop_dir, file)
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)
        print(f"✓ Copied OMOP files to {omop_dir}")

if not os.path.exists(concept_path):
    print("Creating minimal concept table from Synthea source values...")
    print("(In production, use full OMOP vocabulary tables from OHDSI)\n")
    
    concept_data = []
    seen_concepts = set()
    
    # Helper to add concepts from a table
    def add_concepts_from_table(table_name, source_id_col, source_value_col, source_name_col, vocab_id, class_id):
        table_path = os.path.join(omop_dir, f"{table_name}.parquet")
        if not os.path.exists(table_path):
            return
        
        df = pl.read_parquet(table_path)
        if source_id_col not in df.columns or source_value_col not in df.columns:
            return
        
        # Get unique combinations
        unique_df = df.select([source_id_col, source_value_col, source_name_col]).unique()
        
        for row in unique_df.iter_rows():
            concept_id, source_value, source_name = row[0], row[1], row[2] if len(row) > 2 else None
            
            # Skip if concept_id is 0 or we've seen this combination
            if concept_id == 0 or (concept_id, source_value) in seen_concepts:
                continue
            
            seen_concepts.add((concept_id, source_value))
            concept_data.append({
                "concept_id": concept_id,
                "concept_name": source_name or f"Source: {source_value}",
                "concept_code": str(source_value) if source_value else "",
                "vocabulary_id": vocab_id,
                "concept_class_id": class_id,
                "standard_concept": "S",
                "invalid_reason": None
            })
    
    # Collect concepts from various tables
    add_concepts_from_table("condition_occurrence", "condition_source_concept_id", 
                           "condition_source_value", "condition_source_concept_name", 
                           "Synthea", "Condition")
    add_concepts_from_table("drug_exposure", "drug_source_concept_id",
                           "drug_source_value", "drug_source_concept_name",
                           "Synthea", "Drug")
    add_concepts_from_table("procedure_occurrence", "procedure_source_concept_id",
                           "procedure_source_value", "procedure_source_concept_name",
                           "Synthea", "Procedure")
    add_concepts_from_table("measurement", "measurement_source_concept_id",
                           "measurement_source_value", "measurement_source_concept_name",
                           "Synthea", "Measurement")
    
    # Add standard OMOP concept IDs (gender, visit types, etc.)
    standard_concepts = [
        (8507, "Male", "M", "Gender", "Gender"),
        (8532, "Female", "F", "Gender", "Gender"),
        (9201, "Inpatient Visit", "IP", "Visit", "Visit"),
        (9202, "Outpatient Visit", "OP", "Visit", "Visit"),
        (9203, "Emergency Room Visit", "ER", "Visit", "Visit"),
    ]
    
    for concept_id, name, code, vocab, class_id in standard_concepts:
        if concept_id not in {c["concept_id"] for c in concept_data}:
            concept_data.append({
                "concept_id": concept_id,
                "concept_name": name,
                "concept_code": code,
                "vocabulary_id": vocab,
                "concept_class_id": class_id,
                "standard_concept": "S",
                "invalid_reason": None
            })
    
    # Create minimal concept table
    if concept_data:
        concept_df = pl.DataFrame(concept_data)
        # Add required OMOP concept table columns
        concept_df = concept_df.with_columns([
            pl.lit(4180186).alias("language_concept_id"),  # English
            pl.lit(None).cast(pl.Date).alias("valid_start_date"),
            pl.lit(None).cast(pl.Date).alias("valid_end_date"),
        ])
        
        # Ensure proper column order
        required_cols = [
            "concept_id", "concept_name", "concept_code", "vocabulary_id",
            "concept_class_id", "standard_concept", "invalid_reason",
            "language_concept_id", "valid_start_date", "valid_end_date"
        ]
        concept_df = concept_df.select([c for c in required_cols if c in concept_df.columns])
        
        concept_df.write_parquet(concept_path)
        print(f"✓ Created minimal concept table with {len(concept_df)} concepts")
        print(f"  Saved to: {concept_path}")
    else:
        print("⚠️  Could not create concept table - no source values found")
        print("   Note: Synthea OMOP tables may have concept_id=0 for all concepts")
        print("   In this case, you'll need to download OMOP vocabulary tables from OHDSI")
else:
    print(f"✓ Concept table already exists: {concept_path}")

print("\n" + "="*60)
print("Ready to build timelines!")
print("="*60)
print("\nTo build timelines, run:")
print("  timelines_df, eras_df, survey_df = build_omop_timelines(timeline_cfg)")
print("\nThis will:")
print("  1. Load OMOP tables and concept mappings")
print("  2. Build unified event table from all clinical domains")
print("  3. Bin events by time (yearly in this example)")
print("  4. Aggregate into markdown-formatted patient timelines")
print("  5. Save results to parquet files")


✓ Concept table already exists: output/synthea_omop/OMOP/concept.parquet

Ready to build timelines!

To build timelines, run:
  timelines_df, eras_df, survey_df = build_omop_timelines(timeline_cfg)

This will:
  1. Load OMOP tables and concept mappings
  2. Build unified event table from all clinical domains
  3. Bin events by time (yearly in this example)
  4. Aggregate into markdown-formatted patient timelines
  5. Save results to parquet files


In [9]:
# Actually build the timelines
# Uncomment the code below to run the timeline conversion

# Build timelines from OMOP data
timelines_df, eras_df, survey_df = build_omop_timelines(timeline_cfg)

# The function returns:
# - timelines_df: Main timeline table with one row per (person_id, time_bin)
# - eras_df: Era/period table (condition_era, drug_era, observation_period)
# - survey_df: Survey responses (if available)

# Example: View a sample timeline
if timelines_df is not None and len(timelines_df) > 0:
    print(f"\n✓ Built timelines for {timelines_df['person_id'].n_unique()} patients")
    print(f"  Total timeline segments: {len(timelines_df)}")
    print(f"\nSample timeline (first patient, first time bin):")
    print("="*60)
    sample = timelines_df.head(1)
    print(f"Person ID: {sample['person_id'][0]}")
    print(f"Time Bin: {sample['time_bin'][0]}")
    print(f"Age Range: {sample['age_min'][0]:.1f} - {sample['age_max'][0]:.1f} years")
    print(f"\nTimeline Text (first 500 chars):")
    print(sample['timeline_text'][0][:500] + "...")
    print("="*60)

print("To run the timeline conversion, uncomment the code above.")
print("\nThe timeline conversion will:")
print("  • Load all OMOP tables from the configured directory")
print("  • Map concept IDs to concept names using the concept table")
print("  • Build unified event table from all clinical domains")
print("  • Add age at event for each event")
print("  • Bin events by time (yearly, monthly, or by visit)")
print("  • Aggregate into readable patient timelines")
print("  • Save results as parquet files for downstream use")
print("\nTimeline outputs:")
print(f"  • Main timelines: {timeline_cfg.out_parquet}")
print(f"  • Unified events: {timeline_cfg.events_parquet}")
if timeline_cfg.build_eras:
    print(f"  • Eras/periods: {timeline_cfg.eras_parquet}")


🚀 Starting OMOP → timeline pipeline…
🔹 Step 1/8: Loading OMOP dictionary from Google Sheets
📖 Loading OMOP dictionary from Google Sheets…
    → https://docs.google.com/spreadsheets/d/1sPMzOod784PAR0RfjuDezoT15z7w1e0RZsMVlpT2378/export?format=csv&gid=1815943286
✅ Loaded dictionary with 319 rows, 9 columns.
⏱ Step 1 finished in 0.4 s
🔹 Step 2/8: Preparing unified OMOP event table
💡 Loading concept table (concept_id → name/code)…
📦 Building events from raw OMOP tables (will overwrite events cache)…
🧱 Building domain-specific events (lazy)…
🩺 Building condition events from condition_occurrence.parquet…
💊 Building drug events from drug_exposure.parquet…
🧪 Building measurement events from measurement.parquet…
📝 Building observation events from observation.parquet…
🔧 Building procedure events from procedure_occurrence.parquet…
💀 Building death events from death.parquet…
🧱 Concatenating all domain events into a single lazy frame…
📥 Collecting events into memory (may take a bit)…
✅ Events table

SchemaError: datatypes of join keys don't match - `visit_concept_id`: f64 on left does not match `visit_concept_id`: i64 on right (and no other type was available to cast to)

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'sink' <---
SELECT [col("concept_id").alias("visit_concept_id"), col("concept_name").alias("visit_type_name"), col("concept_code").alias("visit_type_code")]
  DF ["concept_id", "concept_name", "concept_code"]; PROJECT */3 COLUMNS

## 12. Additional Resources

- **Synthea GitHub**: https://github.com/synthetichealth/synthea
- **Synthea Wiki**: https://github.com/synthetichealth/synthea/wiki
- **Synthea Documentation**: Comprehensive guides on modules, exporters, and configuration
- **Module Gallery**: Pre-built modules for various conditions and scenarios

## Summary

This tutorial covered:
- ✓ Initializing the SyntheaRunner
- ✓ Running quick simulations
- ✓ Using full configuration options
- ✓ Convenience methods for common use cases
- ✓ Advanced configuration options
- ✓ Working with output files
- ✓ Best practices and tips

You're now ready to generate synthetic patient data for your research!


In [ ]:
# Final example: Quick test run
print("Running final test simulation...")
print("=" * 50)

test_result = runner.run_quick(
    population_size=5,
    state="Massachusetts",
    seed=999,
    output_dir="output/synthea_tutorial_test"
)

if test_result['returncode'] == 0:
    print("\n✓ Tutorial test completed successfully!")
    print(f"✓ Output directory: {test_result['output_dir']}")
    print("\nYou can now explore the generated files in the output directory.")
else:
    print(f"\n✗ Test failed. Check error messages above.")
    print(f"Return code: {test_result['returncode']}")


## ETLSyntheaBuilder

In [4]:
%load_ext rpy2.ipython

INFO:rpy2.situation:cffi mode is CFFI_MODE.ANY
INFO:rpy2.situation:R home found: /home/schilder/.conda/envs/AoU/lib/R
INFO:rpy2.situation:R library path: /usr/local/cuda/lib64:/usr/local/cuda/lib64:
INFO:rpy2.situation:LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/cuda/lib64:
INFO:rpy2.rinterface_lib.embedded:Default options to initialize R: rpy2, --quiet, --no-save
INFO:rpy2.rinterface:Environment variable "PWD" redefined by R and overriding existing variable. Current: "/home/schilder", R: "/home/schilder/projects/AoU/notebooks"
INFO:rpy2.rinterface:R is already initialized. No need to initialize.


In [11]:
%%R
DatabaseConnector::downloadJdbcDrivers("postgresql")


Error in `DatabaseConnector::downloadJdbcDrivers()`:
! The pathToDriver argument must be specified. Consider setting the DATABASECONNECTOR_JAR_FOLDER environment variable, for example in the .Renviron file.
Run `rlang::last_trace()` to see where the error occurred.

Error in DatabaseConnector::downloadJdbcDrivers("postgresql") :


RInterpreterError: Failed to parse and evaluate line 'DatabaseConnector::downloadJdbcDrivers("postgresql")\n'.
R error message: 'Error in DatabaseConnector::downloadJdbcDrivers("postgresql") :'

In [ ]:
%%R


#  devtools::install_github("OHDSI/ETL-Synthea")

 library(ETLSyntheaBuilder)

DatabaseConnector::downloadJdbcDrivers("postgresql")
 # We are loading a version 5.4 CDM into a local PostgreSQL database called "synthea10".
 # The ETLSyntheaBuilder package leverages the OHDSI/CommonDataModel package for CDM creation.
 # Valid CDM versions are determined by executing CommonDataModel::listSupportedVersions().
 # The strings representing supported CDM versions are currently "5.3" and "5.4". 
 # The Synthea version we use in this example is 2.7.0.
 # However, at this time we also support 3.0.0, 3.1.0, 3.2.0 and 3.3.0.
 # Please note that Synthea's MASTER branch is always active and this package will be updated to support
 # future versions as possible.
 # The schema to load the Synthea tables is called "native".
 # The schema to load the Vocabulary and CDM tables is "cdm_synthea10".  
 # The username and pw are "postgres" and "lollipop".
 # The Synthea and Vocabulary CSV files are located in /tmp/synthea/output/csv and /tmp/Vocabulary_20181119, respectively.
 
 # For those interested in seeing the CDM changes from 5.3 to 5.4, please see: http://ohdsi.github.io/CommonDataModel/cdm54Changes.html
 
cd <- DatabaseConnector::createConnectionDetails(
  dbms     = "postgresql", 
  server   = "localhost/synthea10", 
  user     = "postgres", 
  password = "lollipop", 
  port     = 5432, 
#   pathToDriver = "d:/drivers"  
)

cdmSchema      <- "cdm_synthea_omop"
cdmVersion     <- "5.3"
syntheaVersion <- "3.3.0"
syntheaSchema  <- "native"
syntheaFileLoc <- "/home/schilder/projects/AoU/notebooks/output/synthea_omop/output/csv"
vocabFileLoc   <- "/home/schilder/projects/data/OMOP"

ETLSyntheaBuilder::CreateCDMTables(connectionDetails = cd, cdmSchema = cdmSchema, cdmVersion = cdmVersion)
                                     
ETLSyntheaBuilder::CreateSyntheaTables(connectionDetails = cd, syntheaSchema = syntheaSchema, syntheaVersion = syntheaVersion)
                                       
ETLSyntheaBuilder::LoadSyntheaTables(connectionDetails = cd, syntheaSchema = syntheaSchema, syntheaFileLoc = syntheaFileLoc)
                                     
ETLSyntheaBuilder::LoadVocabFromCsv(connectionDetails = cd, cdmSchema = cdmSchema, vocabFileLoc = vocabFileLoc)

ETLSyntheaBuilder::CreateMapAndRollupTables(connectionDetails = cd, cdmSchema = cdmSchema, syntheaSchema = syntheaSchema, cdmVersion = cdmVersion, syntheaVersion = syntheaVersion)

## Optional Step to create extra indices
ETLSyntheaBuilder::CreateExtraIndices(connectionDetails = cd, cdmSchema = cdmSchema, syntheaSchema = syntheaSchema, syntheaVersion = syntheaVersion)
                                    
ETLSyntheaBuilder::LoadEventTables(connectionDetails = cd, cdmSchema = cdmSchema, syntheaSchema = syntheaSchema, cdmVersion = cdmVersion, syntheaVersion = syntheaVersion)

Error in `DatabaseConnector::downloadJdbcDrivers()`:
! The pathToDriver argument must be specified. Consider setting the DATABASECONNECTOR_JAR_FOLDER environment variable, for example in the .Renviron file.
Run `rlang::last_trace()` to see where the error occurred.



Error in DatabaseConnector::downloadJdbcDrivers("postgresql") :


RInterpreterError: Failed to parse and evaluate line '\n\n#  devtools::install_github("OHDSI/ETL-Synthea")\n\n library(ETLSyntheaBuilder)\n\n   DatabaseConnector::downloadJdbcDrivers("postgresql")\n # We are loading a version 5.4 CDM into a local PostgreSQL database called "synthea10".\n # The ETLSyntheaBuilder package leverages the OHDSI/CommonDataModel package for CDM creation.\n # Valid CDM versions are determined by executing CommonDataModel::listSupportedVersions().\n # The strings representing supported CDM versions are currently "5.3" and "5.4". \n # The Synthea version we use in this example is 2.7.0.\n # However, at this time we also support 3.0.0, 3.1.0, 3.2.0 and 3.3.0.\n # Please note that Synthea\'s MASTER branch is always active and this package will be updated to support\n # future versions as possible.\n # The schema to load the Synthea tables is called "native".\n # The schema to load the Vocabulary and CDM tables is "cdm_synthea10".  \n # The username and pw are "post

## Synthea FHIR --> OMOP

In [9]:
# !pyomop --create --vocab ~/projects/data/OMOP/ --input /home/schilder/projects/AoU/notebooks/output/synthea_pediatric/output/fhir

## pyhealth

Get around the hard dep on Python>=1.14:
```
pip install --ignore-requires-python git+https://github.com/sunlabuiuc/PyHealth.git
```
